In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv("fraudTrain.csv.zip")

In [ ]:
# 4. 날짜형 변환
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['dob'] = pd.to_datetime(df['dob'])

df = df.sort_values(['cc_num','trans_date_trans_time'])

from math import radians, sin, cos, sqrt, atan2

def haversine_distance(lat1, lon1, lat2, lon2):

    R = 6371.0

    lat1, lon1, lat2, lon2 = map(
        radians,
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))

    return R * c



# -------------------------------
# 1. 원본 df에서 생성
# -------------------------------

df['is_online'] = df['category'].str.endswith('_net')

# 최근 24시간 고액결제 횟수
df['is_high_amt'] = (df['amt'] >= 500).astype(int)

df_time_idx = df.set_index('trans_date_trans_time')

df['recent_24h_high_amt_count'] = (
    df_time_idx.groupby('cc_num')['is_high_amt']
    .rolling('24h', closed='left', min_periods=0)
    .sum()
    .values
)

# 업종별 최근 7일 사기율
df = df.sort_values(['category', 'trans_date_trans_time'])

df_time_idx = df.set_index('trans_date_trans_time')

df['category_recent_fraud_rate'] = (
    df_time_idx.groupby('category')['is_fraud']
    .rolling('7D', closed='left')
    .mean()
    .values
)

df['category_recent_fraud_rate_missing'] = (
    df['category_recent_fraud_rate']
    .isna()
    .astype(int)
)

df['category_recent_fraud_rate'] = (
    df['category_recent_fraud_rate']
    .fillna(0)
)

# 다시 카드/시간순 정렬
df = df.sort_values(['cc_num', 'trans_date_trans_time']).reset_index(drop=True)

In [ ]:
# -------------------------------
# 2. 오프라인 거래만 추출
# -------------------------------

offline_df = df[
    ~df['category'].isin(['shopping_net', 'misc_net', 'grocery_net'])
].copy()

offline_df = offline_df.sort_values(
    ['cc_num', 'trans_date_trans_time']
)
# -------------------------------
# 3. speed_kmh 생성 (기존 코드)
# -------------------------------

offline_df['prev_lat'] = (
    offline_df.groupby('cc_num')['merch_lat']
    .shift(1)
)

offline_df['prev_long'] = (
    offline_df.groupby('cc_num')['merch_long']
    .shift(1)
)

offline_df['move_distance_km'] = offline_df.apply(
    lambda x: haversine_distance(
        x['prev_lat'],
        x['prev_long'],
        x['merch_lat'],
        x['merch_long']
    )
    if pd.notnull(x['prev_lat']) else np.nan,
    axis=1
)

offline_df['time_diff_sec'] = (
    offline_df.groupby('cc_num')['trans_date_trans_time']
    .diff()
    .dt.total_seconds()
)

offline_df['speed_kmh'] = np.where(
    offline_df['time_diff_sec'] > 0,
    offline_df['move_distance_km'] /
    (offline_df['time_diff_sec'] / 3600),
    np.nan
)

offline_df['speed_kmh'] = offline_df['speed_kmh'].fillna(0)


# -------------------------------
# 4. count_30min 생성
# -------------------------------

offline_df = offline_df.sort_values(
    ['cc_num', 'trans_date_trans_time']
).reset_index(drop=True)

offline_df['count_30min'] = 0

for cc_num, idx in offline_df.groupby('cc_num').groups.items():

    times = (
        offline_df.loc[idx, 'trans_date_trans_time']
        .values.astype('datetime64[s]')
    )

    left = np.searchsorted(
        times,
        times - np.timedelta64(30, 'm')
    )

    right = np.arange(len(times))

    offline_df.loc[idx, 'count_30min'] = right - left + 1


In [ ]:
# -------------------------------
# 5. speed_2 생성
# -------------------------------

def haversine(lat1, lon1, lat2, lon2):
    R = 6371

    phi1, phi2 = np.radians(lat1), np.radians(lat2)

    dphi = np.radians(lat2 - lat1)

    dlambda = np.radians(lon2 - lon1)

    a = (
        np.sin(dphi / 2) ** 2
        + np.cos(phi1)
        * np.cos(phi2)
        * np.sin(dlambda / 2) ** 2
    )

    return 2 * R * np.arcsin(np.sqrt(a))

offline_df['prev_merch_lat'] = (
    offline_df.groupby('cc_num')['merch_lat']
    .shift(1)
)

offline_df['prev_merch_long'] = (
    offline_df.groupby('cc_num')['merch_long']
    .shift(1)
)

offline_df['prev_trans_time'] = (
    offline_df.groupby('cc_num')['trans_date_trans_time']
    .shift(1)
)

offline_df['merchant_shift_km'] = haversine(
    offline_df['prev_merch_lat'],
    offline_df['prev_merch_long'],
    offline_df['merch_lat'],
    offline_df['merch_long']
)

offline_df['time_diff_hr'] = (
    (
        offline_df['trans_date_trans_time']
        - offline_df['prev_trans_time']
    )
    .dt.total_seconds()
    / 3600
)

offline_df['speed_2'] = (
    offline_df['merchant_shift_km']
    / offline_df['time_diff_hr'].clip(lower=1/60)
)



In [ ]:
# -------------------------------
# 6. Repeat3, high_speed
# -------------------------------
# 이동속도 100km/h 이상인 경우만 속도값 유지
offline_df["high_speed"] = np.where(
    offline_df["speed_kmh"] >= 100,
    offline_df["speed_kmh"],
    0
)

# 30분 이내 연속결제 횟수가 3회 이상인 경우만 횟수 유지
offline_df["Repeat3"] = np.where(
    offline_df["count_30min"] >= 3,
    offline_df["count_30min"],
    0
)

# -------------------------------
# 7. 원본 df에 merge
# -------------------------------

offline_features = offline_df[
    [
        'trans_num',
        'speed_kmh',
        'count_30min',
        'Repeat3',
        'high_speed',
        'speed_2'
    ]
]

df = df.merge(
    offline_features,
    on='trans_num',
    how='left'
)

df = df.drop(
    columns=['is_online', 'is_high_amt']
)



In [ ]:
# =====================================================
# 고객별 거래금액 기반 파생변수
# =====================================================

# 시간순 정렬 (반드시 필요)
df = df.sort_values(
    ["cc_num", "trans_date_trans_time"]
).reset_index(drop=True)

# 1. 고객별 과거 누적 평균
df["customer_mean_amt"] = (
    df.groupby("cc_num")["amt"]
    .apply(lambda x: x.shift(1).expanding().mean())
    .reset_index(level=0, drop=True)
)

# 2. 고객별 과거 누적 표준편차
df["customer_std_amt"] = (
    df.groupby("cc_num")["amt"]
    .apply(lambda x: x.shift(1).expanding().std())
    .reset_index(level=0, drop=True)
)

# 3. 현재 거래금액 / 과거 평균
df["amt_ratio_to_mean"] = (
    df["amt"] / df["customer_mean_amt"]
)

# 4. 고객별 거래금액 Z-score
df["amt_zscore_card"] = (
    (df["amt"] - df["customer_mean_amt"])
    / df["customer_std_amt"]
)

# NaN / Inf 처리
df["amt_ratio_to_mean"] = (
    df["amt_ratio_to_mean"]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

df["amt_zscore_card"] = (
    df["amt_zscore_card"]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

# 5. 고객별 이전 거래 횟수
df["customer_transaction_count"] = (
    df.groupby("cc_num")["amt"]
    .apply(lambda x: x.shift(1).expanding().count())
    .reset_index(level=0, drop=True)
    .fillna(0)
)

In [ ]:
# =====================================================
# 거래 시간(hour)
# =====================================================

if "trans_hour" not in df.columns:
    df["trans_hour"] = (
        pd.to_datetime(
            df["trans_date_trans_time"],
            errors="coerce"
        )
        .dt.hour
    )

# NaN -> 0
df["trans_hour"] = df["trans_hour"].fillna(0).astype(int)


# =====================================================
# 거래 당시 나이(age)
# =====================================================

df["age"] = (
    df["trans_date_trans_time"].dt.year
    - df["dob"].dt.year
    - (
        (
            df["trans_date_trans_time"].dt.month
            < df["dob"].dt.month
        )
        |
        (
            (
                df["trans_date_trans_time"].dt.month
                == df["dob"].dt.month
            )
            &
            (
                df["trans_date_trans_time"].dt.day
                < df["dob"].dt.day
            )
        )
    ).astype(int)
)

# NaN -> 0
df["age"] = df["age"].fillna(0).astype(int)


# =====================================================
# 연령대(age_group)
# =====================================================

age_labels = [
    "20세 미만",
    "20대",
    "30대",
    "40대",
    "50대",
    "60세 이상"
]

df["age_group"] = pd.cut(
    df["age"],
    bins=[-np.inf, 20, 30, 40, 50, 60, np.inf],
    labels=age_labels,
    right=False
)

# NaN -> 0
df["age_group"] = (
    df["age_group"]
    .astype(object)
    .fillna(0)
)

In [ ]:
drop_cols = [
    "Unnamed: 0",
    "first",
    "last",
    "gender",
    "street",
    "city",
    "state",
    "zip",
    "city_pop",
    "job",
    "unix_time"
]

df = df.drop(columns=drop_cols)
df['speed_kmh'] = df['speed_kmh'].fillna(0)
df['count_30min'] = df['count_30min'].fillna(0)
df['Repeat3'] = df['Repeat3'].fillna(0)
df['high_speed'] = df['high_speed'].fillna(0)

In [ ]:
df['speed_2'] = df['speed_2'].fillna(0)
df['customer_mean_amt'] = df['customer_mean_amt'].fillna(0)
df['customer_std_amt'] = df['customer_std_amt'].fillna(0)
cols = [
    'speed_kmh',
    'count_30min',
    'Repeat3',
    'high_speed',
    'speed_2',
    'customer_mean_amt',
    'customer_std_amt'
]

df[cols] = df[cols].fillna(0)

# NaN -> 0 처리
fill_zero_cols = [
    'speed_kmh',
    'count_30min',
    'Repeat3',
    'high_speed',
    'speed_2',
    'customer_mean_amt',
    'customer_std_amt'
]

df[fill_zero_cols] = df[fill_zero_cols].fillna(0)

In [ ]:
# 시간순 정렬
# ==========================================
df = df.sort_values("trans_date_trans_time").reset_index(drop=True)


# ==========================================
# 70 : 30
# ==========================================
split_70 = int(len(df) * 0.7)

train_70 = df.iloc[:split_70].copy()
test_30 = df.iloc[split_70:].copy()

train_70.to_csv("fraudTrain_train_70.csv", index=False)
test_30.to_csv("fraudTrain_test_30.csv", index=False)

print("70:30")
print(train_70.shape)
print(test_30.shape)


# ==========================================
# 80 : 20
# ==========================================
split_80 = int(len(df) * 0.8)

train_80 = df.iloc[:split_80].copy()
test_20 = df.iloc[split_80:].copy()

train_80.to_csv("fraudTrain_train_80.csv", index=False)
test_20.to_csv("fraudTrain_test_20.csv", index=False)

print("\n80:20")
print(train_80.shape)
print(test_20.shape)

In [ ]:
df.to_csv(
    'fraudTrain_feature_최종engineering.zip',
    index=False,
    compression='zip'
)

from google.colab import files

files.download('fraudTrain_feature_최종engineering.zip')